# Étape 3 : Évaluation et Calibration

## Pourquoi ne pas utiliser l'Accuracy ?

Avec un ratio 1:150, un modèle naïf qui prédit toujours "Normal" obtient :
$$\text{Accuracy} = \frac{284315}{284807} = 99.83\%$$
Ce score est trompeur — le modèle n'a rien appris.

## Métriques retenues

| Métrique | Formule | Justification |
|----------|---------|---------------|
| **F1-Macro** | $\frac{1}{2}(F1_0 + F1_1)$ | Équilibre précision/rappel sur les deux classes |
| **AUPRC** | $\int_0^1 P(r)\,dr$ | Plus informatif que ROC-AUC pour classes rares |
| **MCC** | $\frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$ | Résumé non-biaisé de toute la matrice de confusion |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
from IPython.display import display, Image

from config import *
from eda import run_eda
from evaluation import (
    evaluate_model, print_metrics, plot_precision_recall_curves,
    plot_reliability_diagram, compare_all_models, calibrate_model
)

# Rechargement des données
eda_data = run_eda(OUTPUT_DIR)
X_train, X_val, X_test = eda_data['X_train'], eda_data['X_val'], eda_data['X_test']
y_train, y_val, y_test = eda_data['y_train'], eda_data['y_val'], eda_data['y_test']

# Chargement des modèles sauvegardés
model_lr  = joblib.load(os.path.join(OUTPUT_DIR, 'model_logistic.pkl'))
model_rf  = joblib.load(os.path.join(OUTPUT_DIR, 'model_rf.pkl'))
model_xgb = joblib.load(os.path.join(OUTPUT_DIR, 'model_xgboost.pkl'))

print('Modèles chargés :', list({'Logistic': model_lr, 'RF': model_rf, 'XGB': model_xgb}.keys()))

## 3.1 Évaluation complète de chaque modèle

In [ ]:
# Calcul des probabilités
y_proba_lr  = model_lr.predict_proba(X_test)[:, 1]
y_proba_rf  = model_rf.predict_proba(X_test)[:, 1]
y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

# Évaluation
metrics_lr  = evaluate_model(y_test, model_lr.predict(X_test),  y_proba_lr,  'Logistic Regression')
metrics_rf  = evaluate_model(y_test, model_rf.predict(X_test),  y_proba_rf,  'Random Forest')
metrics_xgb = evaluate_model(y_test, model_xgb.predict(X_test), y_proba_xgb, 'XGBoost')

print('=== Logistic Regression ===')
print_metrics(metrics_lr)
print('\n=== Random Forest ===')
print_metrics(metrics_rf)
print('\n=== XGBoost ===')
print_metrics(metrics_xgb)

## 3.2 Courbes Precision-Recall et ROC

**Pourquoi AUPRC > ROC-AUC pour les classes déséquilibrées ?**

La courbe ROC trace TPR vs FPR. Avec 284k normaux, même un FPR de 0.01 représente 2843 fausses alarmes — ce qui semble "bon" sur la courbe ROC mais est inacceptable en pratique.

La courbe Precision-Recall mesure directement la qualité de la détection de fraudes.

In [ ]:
pr_results = [
    ('Logistic Regression', y_proba_lr,  metrics_lr),
    ('Random Forest',       y_proba_rf,  metrics_rf),
    ('XGBoost',             y_proba_xgb, metrics_xgb),
]
plot_precision_recall_curves(pr_results, OUTPUT_DIR, y_test, filename='pr_roc_curves.png')
display(Image(filename=os.path.join(OUTPUT_DIR, 'pr_roc_curves.png')))

## 3.3 Calibration — Diagrammes de Fiabilité (Reliability Diagrams)

Un modèle **bien calibré** satisfait :
$$P(Y=1 \mid \hat{p} = p) = p \quad \forall p \in [0,1]$$

Si un modèle prédit $p=0.8$, 80% des transactions correspondantes doivent être des fraudes.

**Expected Calibration Error (ECE)** :
$$\text{ECE} = \sum_{b=1}^{B} \frac{|\mathcal{B}_b|}{n} \left| \text{acc}(\mathcal{B}_b) - \text{conf}(\mathcal{B}_b) \right|$$

- ECE < 0.05 : bien calibré
- ECE ≥ 0.05 : nécessite une calibration

In [ ]:
ece_lr  = plot_reliability_diagram(y_test, y_proba_lr,  'Logistic Regression', OUTPUT_DIR)
ece_rf  = plot_reliability_diagram(y_test, y_proba_rf,  'Random Forest',       OUTPUT_DIR)
ece_xgb = plot_reliability_diagram(y_test, y_proba_xgb, 'XGBoost',             OUTPUT_DIR)

eces = {'Logistic Regression': ece_lr, 'Random Forest': ece_rf, 'XGBoost': ece_xgb}
print('\nExpected Calibration Error (ECE) :')
for name, ece in eces.items():
    status = '✓ bien calibré' if ece < 0.05 else '⚠ recalibration nécessaire'
    print(f'  {name:<25}: ECE={ece:.4f}  {status}')

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, 'calibration_Logistic_Regression.png')))
display(Image(filename=os.path.join(OUTPUT_DIR, 'calibration_Random_Forest.png')))
display(Image(filename=os.path.join(OUTPUT_DIR, 'calibration_XGBoost.png')))

## 3.4 Calibration par Platt Scaling

**Platt Scaling** : ajuste une sigmoïde $\sigma(A \cdot f(x) + B)$ sur les sorties du modèle.
- Paramétrique, entraîné par maximum de vraisemblance
- Efficace pour les modèles avec distribution en forme de S (SVM, boost)

**Isotonic Regression** : transformation monotone non-paramétrique.
- Plus flexible mais risque d'overfitting sur petits datasets

In [ ]:
worst_model_name = max(eces, key=eces.get)
worst_ece = eces[worst_model_name]
model_map = {'Logistic Regression': model_lr, 'Random Forest': model_rf, 'XGBoost': model_xgb}

print(f'Modèle le plus mal calibré : {worst_model_name} (ECE={worst_ece:.4f})')

if worst_ece > 0.05:
    print(f'Application du Platt Scaling...')
    cal_result = calibrate_model(
        model_map[worst_model_name],
        X_train, y_train, X_test, y_test,
        method='sigmoid', cv=3,
        output_dir=OUTPUT_DIR,
        model_name=worst_model_name
    )
    display(Image(filename=os.path.join(OUTPUT_DIR, f'calibration_platt_scaling.png')))
else:
    print('Tous les modèles sont bien calibrés — Platt Scaling non nécessaire.')

## 3.5 Comparaison finale des modèles

In [ ]:
all_metrics = [metrics_lr, metrics_rf, metrics_xgb]
df_compare = compare_all_models(all_metrics, OUTPUT_DIR)

print('\nTableau comparatif :')
display(df_compare)

print('\nMeilleur modèle par AUPRC :')
best = df_compare.loc[df_compare['AUPRC'].astype(float).idxmax()]
print(f'  {best["Model"]} — AUPRC={best["AUPRC"]}, MCC={best["MCC"]}')

display(Image(filename=os.path.join(OUTPUT_DIR, 'model_comparison.png')))

In [ ]:
# Résumé CSV
results_csv = pd.read_csv(os.path.join(OUTPUT_DIR, 'results_summary.csv'))
print('Résultats sauvegardés :')
display(results_csv.style.highlight_max(subset=['F1_Macro','AUPRC','MCC'], color='lightgreen'))

## Résumé de l'étape 3

### Justification des métriques
1. **F1-Macro** : Évite le biais de l'accuracy, traite les deux classes équitablement
2. **AUPRC** : Mesure directe de la capacité à trouver les fraudes sans fausses alarmes. Baseline = 0.17% (aléatoire)
3. **MCC** : Seule métrique non-biaisée intégrant les 4 cases de la matrice de confusion. Score 0 = prédiction aléatoire

### Calibration
La calibration est critique en fraude bancaire : un score de 0.8 doit vraiment signifier 80% de probabilité de fraude pour guider la décision de blocage ou d'investigation manuelle.